<a href="https://colab.research.google.com/github/paulosantosps/Case_Rank/blob/main/Camada_Bronze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**SCRIPT PARA GERAÇÃO DA CAMADA BRONZE**

**1º Comandos de Sistema para instalação das bibliotecas que permite a conexão com o Banco de Dados no PostgreSQL/Supabase:**

In [1]:
!pip install sqlalchemy psycopg2-binary openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 28.7 MB/s eta 0:00:00


**2º Aplicando a Biblioteca Pandas para armazenar as tabelas do arquivo XLSX em Dataframes**

In [33]:
import pandas as pd

arquivo = "/RankMyAPP - Case Sênior.xlsx"  #arquivo XLSX importado manualmente no Notebook

install = pd.read_excel(arquivo, sheet_name="Report_ecommerce_install")
event  = pd.read_excel(arquivo, sheet_name="Report_ecommerce_event")
fat  = pd.read_excel(arquivo, sheet_name="Compras_faturadas")

**3º Verificação do correto carregamento das bases de dados**

In [12]:
print(install.shape, event.shape, fat.shape)

(25894, 23) (12275, 22) (13636, 11)


**4° Padronização dos Nomes das Colunas para melhor utilização no SQL**

In [35]:
import re, unicodedata #bibliotecas para padronização dos nomes das colunas

#criando função REGEX para substituir acentos e espaçamentos por '_' para facilitar a utilozação no SQL
def snake_case(aux):
    aux = unicodedata.normalize("NFKD", str(aux)).encode("ascii", "ignore").decode()
    return re.sub(r"[^0-9a-zA-Z]+", "_", aux).strip("_").lower()

In [37]:
for df in (install, event, fat):
    df.columns = [snake_case(c) for c in df.columns]
    df["_data_carregamento"] = pd.Timestamp.now()

**5° Estabelecendo conexão com o PostgreSQL/Supabase**

In [38]:
from google.colab import userdata
from sqlalchemy import create_engine, text #função 'text' permite executar comandos SQL

engine = create_engine(userdata.get("DATABASE_URL")) #Chamada da Chave de acesso ao Banco de dados PostgreSQL (previamente cadastrada)

with engine.connect() as conectar:
    print(conectar.execute(text("select version()")).scalar()) #Estabelecendo conexão com o banco de dados

PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit


**6° Criando o schema e carregaando os Dataframes**

In [39]:
with engine.begin() as conectar:
    conectar.execute(text("create schema if not exists bronze")) #criação do Schema no PostgreSQL

tabelas = {"install_brz": install, "event_brz": event, "faturadas_brz": fat} #definindo o nome das tabelas no PostgreSQL

for nome, df in tabelas.items():
    df.to_sql(nome, engine, schema="bronze", if_exists="replace",
              index=False, method="multi", chunksize=1000)  #carrega os fataframs no banco de dados em blocos
    print("carregada:", nome)  #comando de verificação

carregada: install_brz
carregada: event_brz
carregada: faturadas_brz


**7° Validação de Carregamento no PostgreSQL/Supabase**

In [41]:
with engine.connect() as conectar:
    for nome in tabelas:
        n = conectar.execute(text(f"select count(*) from bronze.{nome}")).scalar()
        print(nome, n)

install_brz 25894
event_brz 12275
faturadas_brz 13636


In [42]:
print(install.shape, event.shape, fat.shape)

(25894, 25) (12275, 24) (13636, 13)


*Como o numero de linhas dos dataframes e das tabelas no banco de dados conferem, validamos o correto carregamento dos dados!*